In [ ]:


import threading
import traceback
import tkinter as tk
from tkinter import ttk, messagebox, filedialog

import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk

from cnt_wakefields_nlayer import MWCNT as MWCNT_Wake, wz_general_nlayer, wakefields_closed
from cnt_wakefields import C_AU, BOHR_TO_NM, _g, NM_TO_AU, N_G_AU

EV_PER_AU = 27.211386245988  # hartree -> eV, para pintar la energia del plasmon


# --------------------------------------------------------------------- utils
def parse_float(entry, name, default=None):
    txt = entry.get().strip()
    if txt == "" and default is not None:
        return default
    try:
        return float(txt)
    except ValueError:
        raise ValueError(f"'{name}' no es un numero valido: '{txt}'")


def parse_int(entry, name, default=None):
    txt = entry.get().strip()
    if txt == "" and default is not None:
        return default
    try:
        return int(float(txt))
    except ValueError:
        raise ValueError(f"'{name}' no es un entero valido: '{txt}'")


def parse_list_float(entry, name):
    txt = entry.get().strip()
    if txt == "":
        return []
    out = []
    for piece in txt.split(","):
        piece = piece.strip()
        if piece == "":
            continue
        try:
            out.append(float(piece))
        except ValueError:
            raise ValueError(f"'{name}' contiene un valor no numerico: '{piece}'")
    return out


def parse_list_int(entry, name):
    return [int(round(x)) for x in parse_list_float(entry, name)]


COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
          "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan"]
LINESTYLES = ["-", "--", "-.", ":"]


# --------------------------------------------------------------- fila pared
class WallRow:
    """Una fila editable de la tabla de paredes: a (nm), n0/n_g.
    Compartida por el calculo de wakefields y el de dispersion: son
    las paredes del MISMO nanotubo."""

    def __init__(self, parent, on_remove, a=0.36, n0=1.0):
        self.frame = ttk.Frame(parent)
        self.a_var = tk.StringVar(value=str(a))
        self.n0_var = tk.StringVar(value=str(n0))

        ttk.Entry(self.frame, textvariable=self.a_var, width=12).grid(row=0, column=0, padx=2)
        ttk.Entry(self.frame, textvariable=self.n0_var, width=12).grid(row=0, column=1, padx=2)
        ttk.Button(self.frame, text="Quitar", width=8,
                   command=lambda: on_remove(self)).grid(row=0, column=2, padx=4)

        self.frame.pack(fill="x", pady=1)

    def destroy(self):
        self.frame.destroy()

    def values(self):
        try:
            a = float(self.a_var.get())
            n0 = float(self.n0_var.get())
        except ValueError:
            raise ValueError("Todas las paredes necesitan valores numericos (a, n0/n_g).")
        return a, n0


# ------------------------------------------- MWCNT (dispersion), N paredes
# Generaliza las Ecs. (16)/(20)-(22) del paper a N paredes: las omega_i(m,k)
# acopladas son los autovalores de la matriz K(m,k) construida a partir de
# la Ec. (7)/(12)/(9)/(21) del capitulo, planteada alli mismo para N
# generico antes de particularizar a N=1 (SWCNT) y N=2 (DWCNT). Reproduce
# exactamente SWCNT._omega2 (N=1) y DWCNT._omega_pm2 (N=2) de
# cnt_wakefields.py -- validado numericamente.
class MWCNT_Disp:
    """Nanotubo de N paredes para el calculo de dispersion.

    Parameters
    ----------
    a_nm : lista de radios (nm). N=1 -> SWCNT, N=2 -> DWCNT, N>=3 -> MWCNT.
    n0_over_ng : densidad superficial de cada pared en unidades de
        n_g = 4*0.107 (a.u.). Escalar (misma densidad en todas las
        paredes) o lista de la misma longitud que a_nm.
    beta : coeficiente del termino de Von Weizsacker (beta=1/4 por defecto).
    """

    def __init__(self, a_nm, n0_over_ng=1.0, beta=0.25):
        a_nm = np.atleast_1d(np.asarray(a_nm, dtype=float))
        order = np.argsort(a_nm)
        self.a = a_nm[order] * NM_TO_AU
        self.N = len(self.a)

        n0_over_ng = np.atleast_1d(np.asarray(n0_over_ng, dtype=float))
        if len(n0_over_ng) == 1 and self.N > 1:
            n0_over_ng = np.repeat(n0_over_ng, self.N)
        self.n0 = n0_over_ng[order] * N_G_AU

        self.beta = beta
        self.alpha = np.pi * self.n0  # alpha_j por pared, array (N,)

    def _q2(self, j, m, k):
        return k**2 + m**2 / self.a[j] ** 2

    def omega_j2(self, j, m, k):
        """omega_j^2(m,k) de la pared j sola (Eq. 21 generalizada)."""
        q2 = self._q2(j, m, k)
        Gjj = self.n0[j] * self.a[j] * q2 * _g(self.a[j], self.a[j], m, k)
        return self.alpha[j] * q2 + self.beta * q2 ** 2 + Gjj

    def G_jl(self, j, l, m, k):
        """Acoplo electrostatico  entre la pared j 
        y la pared l (fuente de la densidad perturbada)."""
        ql2 = self._q2(l, m, k)
        return self.n0[l] * self.a[l] * ql2 * _g(self.a[j], self.a[l], m, k)

    def K_matrix(self, m, k):
        N = self.N
        K = np.empty((N, N))
        for j in range(N):
            K[j, j] = self.omega_j2(j, m, k)
        for j in range(N):
            for l in range(N):
                if l != j:
                    K[j, l] = self.G_jl(j, l, m, k)
        return K

    def branches(self, m, k):
        """omega_i^2(m,k) para i=1..N (orden ascendente), k escalar."""
        K = self.K_matrix(m, k)
        w2 = np.linalg.eigvals(K)
        return np.sort(w2.real)

    def branches_array(self, m, k_array):
        """Igual que branches() pero vectorizado sobre un array de k."""
        k_array = np.asarray(k_array, dtype=float)
        out = np.empty((len(k_array), self.N))
        for i, k in enumerate(k_array):
            out[i, :] = self.branches(m, k)
        return out


# ------------------------------------------------------------------- app
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Nanotubos de carbono (MWCNT): wakefields y dispersion")
        self.geometry("1400x880")

        self.wall_rows = []
        self.figures = []

        self._build_layout()
        self._add_wall_row(0.36, 1.0)

    # ---------------------------------------------------------- layout
    def _build_layout(self):
        outer = ttk.Frame(self)
        outer.pack(fill="both", expand=True)

        left_container = ttk.Frame(outer, width=400)
        left_container.pack(side="left", fill="y")
        left_container.pack_propagate(False)

        f_run = ttk.Frame(left_container)
        f_run.pack(side="bottom", fill="x", padx=8, pady=6)
        ttk.Separator(left_container, orient="horizontal").pack(side="bottom", fill="x")
        self.btn_run = ttk.Button(f_run, text="Ejecutar", command=self._on_run)
        self.btn_run.pack(fill="x", pady=4)
        self.btn_save = ttk.Button(f_run, text="Guardar figuras como PNG...",
                                    command=self._on_save, state="disabled")
        self.btn_save.pack(fill="x")
        self.progress = ttk.Progressbar(f_run, mode="indeterminate")
        self.progress.pack(fill="x", pady=4)
        self.status_var = tk.StringVar(value="Listo.")
        ttk.Label(f_run, textvariable=self.status_var, foreground="#444",
                  wraplength=360, justify="left").pack(fill="x")

        self.left_canvas = tk.Canvas(left_container, borderwidth=0, highlightthickness=0)
        vscroll = ttk.Scrollbar(left_container, orient="vertical", command=self.left_canvas.yview)
        self.left = ttk.Frame(self.left_canvas)
        self.left.bind("<Configure>",
                        lambda e: self.left_canvas.configure(scrollregion=self.left_canvas.bbox("all")))
        self.left_canvas.create_window((0, 0), window=self.left, anchor="nw")
        self.left_canvas.configure(yscrollcommand=vscroll.set)
        vscroll.pack(side="left", fill="y")
        self.left_canvas.pack(side="left", fill="both", expand=True)

        right_container = ttk.Frame(outer)
        right_container.pack(side="left", fill="both", expand=True)
        self.right_canvas = tk.Canvas(right_container, borderwidth=0, highlightthickness=0)
        right_vscroll = ttk.Scrollbar(right_container, orient="vertical", command=self.right_canvas.yview)
        self.right = ttk.Frame(self.right_canvas)
        self.right.bind("<Configure>", self._update_right_scrollregion)
        self.right_canvas.create_window((0, 0), window=self.right, anchor="nw")
        self.right_canvas.configure(yscrollcommand=right_vscroll.set)
        self.right_canvas.pack(side="left", fill="both", expand=True)
        right_vscroll.pack(side="right", fill="y")

        pad = dict(padx=8, pady=4)

        # ---- Paredes del nanotubo (COMPARTIDAS) ----
        f_walls = ttk.LabelFrame(self.left, text="Paredes del nanotubo (radios)")
        f_walls.pack(fill="x", **pad)
        hdr = ttk.Frame(f_walls)
        hdr.pack(fill="x")
        for txt, w in (("a (nm)", 12), ("n0 / n_g", 12)):
            ttk.Label(hdr, text=txt, width=w).pack(side="left", padx=2)
        self.walls_frame = ttk.Frame(f_walls)
        self.walls_frame.pack(fill="x")
        ttk.Button(f_walls, text="+ Anadir pared",
                   command=lambda: self._add_wall_row(
                       round(0.36 * (len(self.wall_rows) + 1), 3), 1.0)
                   ).pack(pady=4)
        ttk.Label(f_walls, text="N=1 -> SWCNT, N=2 -> DWCNT, N>=3 -> MWCNT.\n"
                                 "El orden no importa: se ordenan por radio.\n"
                                 "Se usan tanto para la amplitud del wakefield \n"
                                 "como para la dispersion: es el mismo nanotubo.",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        ttk.Separator(self.left, orient="horizontal").pack(fill="x", padx=8, pady=6)
        ttk.Label(self.left, text="WAKEFIELD", font=("", 10, "bold")).pack(anchor="w", padx=8)

        # ---- Driver ----
        f_drv = ttk.LabelFrame(self.left, text="Carga excitadora (driver)")
        f_drv.pack(fill="x", **pad)
        self.e_v = self._labeled_entry(f_drv, "v / c", "0.05")
        self.e_Q = self._labeled_entry(f_drv, "Q (carga, u.a.)", "1.0")
        self.e_r0 = self._labeled_entry(f_drv, "r0 (nm, radial)", "0.0")
        self.e_phi0 = self._labeled_entry(f_drv, "phi0 (grados)", "0.0")
        ttk.Label(f_drv, text="r0=0 (on-axis) excita solo el modo m=0.",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Observacion ----
        f_obs = ttk.LabelFrame(self.left, text="Punto de observacion")
        f_obs.pack(fill="x", **pad)
        self.e_robs = self._labeled_entry(f_obs, "r (nm) [vacio = 0, eje]", "")
        self.e_phiobs = self._labeled_entry(f_obs, "phi (grados)", "0.0")
        ttk.Label(f_obs, text="r=0 (eje) es el caso mas habitual \n"
                               "Los resultados dejan de ser válidos coherentes con el LHM\n"
                               "cuando la distancia de la carga al tubo es menor que la distancia interatómica \n"
                               "interatómica del material(14 nm)",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Friccion ----
        f_gamma = ttk.LabelFrame(self.left, text="Friccion")
        f_gamma.pack(fill="x", **pad)
        self.e_gamma = self._labeled_entry(f_gamma, "gamma / Omega", "1e-3")
        ttk.Label(f_gamma, text="Omega = sqrt(4*pi*n0_1/a_1), de la pared\n"
                                 "más interna. gamma se aplica igual a\n"
                                 "todas las paredes.",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Modos y malla en k (wakefield) ----
        f_mesh = ttk.LabelFrame(self.left, text="Modos angulares y malla en k ")
        f_mesh.pack(fill="x", **pad)
        self.e_mmax = self._labeled_entry(f_mesh, "m_max (modos)", "6")
        self.e_kmax = self._labeled_entry(f_mesh, "k_max (u.a.)", "3.0")
        ttk.Label(f_mesh, text="m_max es irrelevante si r0=0 (solo\n"
                                 "sobrevive m=0). k_max acota la búsqueda\n"
                                 "de resonancias y la integral en k",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Rango zeta ----
        f_zeta = ttk.LabelFrame(self.left, text="Rango zeta (nm")
        f_zeta.pack(fill="x", **pad)
        self.e_zeta_min = self._labeled_entry(f_zeta, "zeta min", "-20.0")
        self.e_zeta_max = self._labeled_entry(f_zeta, "zeta max", "0.5")

        # ---- Curvas de wakefield a mostrar ----
        f_curves = ttk.LabelFrame(self.left, text="Amplitudes de Wakefield a mostrar")
        f_curves.pack(fill="x", **pad)
        self.var_show_wake = tk.BooleanVar(value=True)
        ttk.Checkbutton(f_curves, text="Mostrar grafica de amplitud del Wakefield",
                         variable=self.var_show_wake,
                         command=self._on_toggle_wake).pack(anchor="w", pady=(0, 4))
        ttk.Separator(f_curves, orient="horizontal").pack(fill="x", pady=2)
        self.var_total = tk.BooleanVar(value=True)
        self.var_re = tk.BooleanVar(value=False)
        self.var_im = tk.BooleanVar(value=False)
        self.var_closed = tk.BooleanVar(value=True)
        self.chk_total = ttk.Checkbutton(f_curves, text="Total (Re+Im, friccion finita)", variable=self.var_total)
        self.chk_re = ttk.Checkbutton(f_curves, text="Parte Re", variable=self.var_re)
        self.chk_im = ttk.Checkbutton(f_curves, text="Parte Im", variable=self.var_im)
        self.chk_closed = ttk.Checkbutton(f_curves, text="Limite cerrado gamma->0+ (exacto)", variable=self.var_closed)
        for chk in (self.chk_total, self.chk_re, self.chk_im, self.chk_closed):
            chk.pack(anchor="w")

        ttk.Separator(self.left, orient="horizontal").pack(fill="x", padx=8, pady=6)
        ttk.Label(self.left, text="DISPERSION", font=("", 10, "bold")).pack(anchor="w", padx=8)

        # ---- Modos y rango de k (dispersion) ----
        f_dmodes = ttk.LabelFrame(self.left, text="Modos angulares y rango de k (dispersion)")
        f_dmodes.pack(fill="x", **pad)
        self.e_dmmax = self._labeled_entry(f_dmodes, "m maximo (dispersion)", "2")
        ttk.Label(f_dmodes, text="Se muestran todos los modos m = 0..m_max.",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(0, 4))
        self.e_dkmin = self._labeled_entry(f_dmodes, "k min (1/nm)", "0.001")
        self.e_dkmax = self._labeled_entry(f_dmodes, "k max (1/nm)", "8.0")
        
        self.e_dnpts = self._labeled_entry(f_dmodes, "num. puntos en k", "400")

        # ---- Velocidades (rectas k*v) ----
        f_vel = ttk.LabelFrame(self.left, text="Velocidades del driver (rectas k v)")
        f_vel.pack(fill="x", **pad)
        self.e_vlist = self._labeled_entry(f_vel, "v/c (lista, coma)", "0.025, 0.05, 0.2")
        ttk.Label(f_vel, text="Se dibujan como rectas kv en las gráficas\n"
                               "(A), (B) y (C): donde cruzan una curva de\n"
                               "dispersión hay resonancia (kv=omega_m(k)).",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Grafica de dispersion (una sola, unificada) ----
        f_dplots = ttk.LabelFrame(self.left, text="Grafica de dispersion")
        f_dplots.pack(fill="x", **pad)
        self.var_show_disp = tk.BooleanVar(value=True)
        ttk.Checkbutton(f_dplots, text="Mostrar grafica de dispersion",
                         variable=self.var_show_disp,
                         command=self._on_toggle_disp).pack(anchor="w", pady=(0, 4))
        ttk.Separator(f_dplots, orient="horizontal").pack(fill="x", pady=2)
        self.var_show_walls_ref = tk.BooleanVar(value=True)
        self.chk_walls_ref = ttk.Checkbutton(
            f_dplots, variable=self.var_show_walls_ref,
            text="Incluir paredes sueltas, sin acoplo (referencia,\n"
                 "lineas finas grises, estilo Fig. 3)")
        self.chk_walls_ref.pack(anchor="w", padx=4, pady=2)

        self._bind_mousewheel(self.left, self.left_canvas)
        self.e_r0.bind("<KeyRelease>", self._on_r0_change)
        self._on_r0_change()

    def _update_right_scrollregion(self, event=None):
        self.right_canvas.configure(scrollregion=self.right_canvas.bbox("all"))

    def _bind_mousewheel(self, widget, target_canvas):
        def _on_wheel(event):
            if getattr(event, "num", None) == 4:
                target_canvas.yview_scroll(-1, "units")
            elif getattr(event, "num", None) == 5:
                target_canvas.yview_scroll(1, "units")
            elif getattr(event, "delta", 0):
                target_canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")
        widget.bind("<MouseWheel>", _on_wheel)
        widget.bind("<Button-4>", _on_wheel)
        widget.bind("<Button-5>", _on_wheel)
        for child in widget.winfo_children():
            self._bind_mousewheel(child, target_canvas)

    def _labeled_entry(self, parent, label, default):
        row = ttk.Frame(parent)
        row.pack(fill="x", padx=4, pady=2)
        ttk.Label(row, text=label, width=20).pack(side="left")
        e = ttk.Entry(row, width=14)
        e.insert(0, default)
        e.pack(side="left", fill="x", expand=True)
        return e

    def _add_wall_row(self, a, n0):
        row = WallRow(self.walls_frame, self._remove_wall_row, a, n0)
        self.wall_rows.append(row)

    def _remove_wall_row(self, row):
        if len(self.wall_rows) <= 1:
            messagebox.showwarning("Aviso", "Debe quedar al menos una pared.")
            return
        row.destroy()
        self.wall_rows.remove(row)

    def _on_toggle_wake(self):
        state = "normal" if self.var_show_wake.get() else "disabled"
        for chk in (self.chk_total, self.chk_re, self.chk_im, self.chk_closed):
            chk.configure(state=state)

    def _on_toggle_disp(self):
        state = "normal" if self.var_show_disp.get() else "disabled"
        self.chk_walls_ref.configure(state=state)

    def _on_r0_change(self, event=None):
        """m_max no afecta al resultado si r0=0 (solo sobrevive m=0)"""
        txt = self.e_r0.get().strip()
        try:
            r0 = float(txt)
        except ValueError:
            r0 = None
        if r0 == 0.0:
            self.e_mmax.configure(state="enabled")
        else:
            self.e_mmax.configure(state="normal")

    # ---------------------------------------------------------- ejecutar
    def _on_run(self):
        try:
            params = self._collect_params()
        except ValueError as e:
            messagebox.showerror("Parametro invalido", str(e))
            return

        self.btn_run.configure(state="disabled")
        self.btn_save.configure(state="disabled")
        self.progress.start(10)
        self.status_var.set("Calculando...")

        threading.Thread(target=self._run_worker, args=(params,), daemon=True).start()

    def _collect_params(self):
        a_list, n0_list = [], []
        for row in self.wall_rows:
            a, n0 = row.values()
            a_list.append(a)
            n0_list.append(n0)
        if len(a_list) == 0:
            raise ValueError("Anade al menos una pared.")

        need_wake = self.var_show_wake.get()
        show_wake = dict(total=self.var_total.get(), re=self.var_re.get(),
                          im=self.var_im.get(), closed=self.var_closed.get())
        if need_wake and not any(show_wake.values()):
            raise ValueError("Has marcado 'Mostrar gráfica de amplitud de Wakefield' pero ninguna curva "
                              "(Total/Re/Im/cerrado). Marca al menos una o desactiva esa casilla.")

        need_disp = self.var_show_disp.get()
        show_walls_ref = self.var_show_walls_ref.get()

        if not need_wake and not need_disp:
            raise ValueError("Activa al menos una sección (Wakefield o dispersión).")

        p = dict(a_list=a_list, n0_list=n0_list,
                  need_wake=need_wake, need_disp=need_disp,
                  show_wake=show_wake, show_walls_ref=show_walls_ref)

        if need_wake:
            v_over_c = parse_float(self.e_v, "v/c")
            Q = parse_float(self.e_Q, "Q")
            r0_nm = parse_float(self.e_r0, "r0")
            phi0_deg = parse_float(self.e_phi0, "phi0")

            r_obs_txt = self.e_robs.get().strip()
            r_obs_nm = float(r_obs_txt) if r_obs_txt else 0.0

            phi_obs_deg = parse_float(self.e_phiobs, "phi obs")

            gamma_over_Omega = parse_float(self.e_gamma, "gamma/Omega")

            m_max = parse_int(self.e_mmax, "m_max")
            if r0_nm == 0.0:
                m_max = 0  # r0=0 -> solo sobrevive m=0 
            k_max = parse_float(self.e_kmax, "k_max")
            if m_max < 0:
                raise ValueError("m_max debe ser >= 0.")
            if k_max <= 0:
                raise ValueError("k_max debe ser > 0.")

            zeta_min = parse_float(self.e_zeta_min, "zeta min")
            zeta_max = parse_float(self.e_zeta_max, "zeta max")
            if zeta_max <= zeta_min:
                raise ValueError("zeta max debe ser mayor que zeta min.")

            p.update(v_over_c=v_over_c, Q=Q, r0_nm=r0_nm, phi0=np.deg2rad(phi0_deg),
                      r_obs_nm=r_obs_nm, phi_obs=np.deg2rad(phi_obs_deg),
                      gamma_over_Omega=gamma_over_Omega, m_max=m_max, k_max=k_max,
                      zeta=np.linspace(zeta_min, zeta_max, 1200))

        if need_disp:
            d_m_max = parse_int(self.e_dmmax, "m maximo (dispersion)")
            if d_m_max < 0:
                raise ValueError("m maximo (dispersion) debe ser >= 0.")
            m_list = list(range(0, d_m_max + 1))

            k_min_inv_nm = parse_float(self.e_dkmin, "k min")
            k_max_inv_nm = parse_float(self.e_dkmax, "k max")
            if k_max_inv_nm <= k_min_inv_nm:
                raise ValueError("k max (dispersion) debe ser mayor que k min.")
            npts = int(parse_float(self.e_dnpts, "num. puntos"))
            if npts < 10:
                raise ValueError("num. puntos en k debe ser >= 10.")

            v_list = parse_list_float(self.e_vlist, "v/c (lineas kv)")

            p.update(d_m_list=sorted(set(m_list)), d_k_min_inv_nm=k_min_inv_nm,
                      d_k_max_inv_nm=k_max_inv_nm, d_npts=npts, d_v_list=v_list)

        return p

    def _run_worker(self, p):
        try:
            tube_wake = res = Wz0lim = tube_disp = None

            if p["need_wake"]:
                tube_wake = MWCNT_Wake(a_nm=p["a_list"], v_over_c=p["v_over_c"],
                                        n0_over_ng=p["n0_list"], Q=p["Q"],
                                        r0_nm=p["r0_nm"], phi0=p["phi0"])
                res = wz_general_nlayer(tube_wake, p["r_obs_nm"], p["zeta"],
                                         p["gamma_over_Omega"], phi=p["phi_obs"],
                                         m_max=p["m_max"], k_max=p["k_max"])
                if p["show_wake"]["closed"]:
                    Wz0lim = wakefields_closed(tube_wake, p["r_obs_nm"], p["zeta"],
                                                phi=p["phi_obs"], m_max=p["m_max"],
                                                k_max=p["k_max"])

            if p["need_disp"]:
                tube_disp = MWCNT_Disp(a_nm=p["a_list"], n0_over_ng=p["n0_list"])

            self.after(0, self._on_run_done, tube_wake, tube_disp, p, res, Wz0lim, None)
        except Exception as exc:
            tb = traceback.format_exc()
            self.after(0, self._on_run_done, None, None, None, None, None, (exc, tb))

    def _on_run_done(self, tube_wake, tube_disp, p, res, Wz0lim, error):
        self.progress.stop()
        self.btn_run.configure(state="normal")

        if error is not None:
            exc, tb = error
            self.status_var.set(f"Error: {exc}")
            messagebox.showerror("Error durante el cálculo", f"{exc}\n\nDetalles en consola.")
            print(tb)
            return

        self.status_var.set("Calculo completado.")
        self.btn_save.configure(state="normal")
        self._draw_results(tube_wake, tube_disp, p, res, Wz0lim)

    # ------------------------------------------------------------ dibujo
    def _tube_label(self, N, a_list):
        if N == 1:
            return f"SWCNT (N=1), $a={a_list[0]:g}$ nm"
        elif N == 2:
            return f"DWCNT (N=2), a=[{', '.join(f'{x:g}' for x in sorted(a_list))}] nm"
        radii_str = ", ".join(f"{x:g}" for x in sorted(a_list))
        return f"MWCNT (N={N}), a=[{radii_str}] nm"

    def _draw_results(self, tube_wake, tube_disp, p, res, Wz0lim):
        for w in self.right.winfo_children():
            w.destroy()
        self.figures = []

        if p["need_wake"]:
            self._plot_wakefield(tube_wake, p, res, Wz0lim)

        if p["need_disp"]:
            k_inv_nm = np.linspace(p["d_k_min_inv_nm"], p["d_k_max_inv_nm"], p["d_npts"])
            k_au = k_inv_nm * BOHR_TO_NM
            label = self._tube_label(tube_disp.N, p["a_list"])
            self._plot_dispersion(tube_disp, p, k_inv_nm, k_au, label)

        self._bind_mousewheel(self.right, self.right_canvas)
        self.right.update_idletasks()
        self._update_right_scrollregion()

    def _new_canvas(self, fig, name):
        holder = ttk.Frame(self.right)
        holder.pack(fill="both", expand=True, pady=(4, 12))
        canvas = FigureCanvasTkAgg(fig, master=holder)
        canvas.draw()
        toolbar = NavigationToolbar2Tk(canvas, holder)
        toolbar.update()
        canvas.get_tk_widget().pack(fill="both", expand=True)
        self.figures.append((fig, name))

    def _plot_wakefield(self, tube, p, res, Wz0lim):
        label = self._tube_label(tube.N, p["a_list"])
        show = p["show_wake"]
        zeta = p["zeta"]

        fig = plt.Figure(figsize=(7.5, 5.0))
        ax = fig.subplots(1, 1)
        ax.axhline(0, color="k", lw=0.6, alpha=0.5)

        if show["total"]:
            ax.plot(zeta, res["total"], color="k", lw=1.8, label=r"$W_{z}(\mathrm{total})$")
        if show["re"]:
            ax.plot(zeta, res["Wz1"], color="tab:orange", lw=1.5, ls="--", label=r"$W_{z,\mathrm{Re}}$")
        if show["im"]:
            ax.plot(zeta, res["Wz2"], color="tab:blue", lw=1.3, ls="-.", label=r"$W_{z,\mathrm{Im}}$")
        if show["closed"] and Wz0lim is not None:
            ax.plot(zeta, Wz0lim, color="red", lw=1.4, ls=":", label=r"$W_{z}\ (\gamma \to 0^{+})$")

        ax.set_xlabel(r"$\zeta$ (nm)")
        ax.set_ylabel(r"$W_z$ (GV/m)")
        ax.set_xlim(zeta.min(), zeta.max())
        ax.legend(loc="best", frameon=True, fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_title(
            rf"Wakefields -- {label}, $v={p['v_over_c']:g}c$, $r={p['r_obs_nm']:g}$ nm, "
            rf"$\gamma={p['gamma_over_Omega']:g}\,\Omega$"
        )
        fig.tight_layout()
        self._new_canvas(fig, "wakefield.png")

    def _to_eV(self, w2_au):
        return np.sqrt(np.clip(w2_au, 0, None)) * EV_PER_AU

    def _plot_dispersion(self, tube, p, k_inv_nm, k_au, label):
        """Gráfica única de dispersión:
        - Las curvas/ramas de dispersión siempre en NEGRO, una línea
          por modo m con su propio trazo (solido/rayas/puntos...). Si
          hay varias paredes acopladas (N>=2), cada modo m tiene N
          ramas ; comparten el mismo trazo negro de su m, sin
          diferenciar rama por color .
        - Las paredes sueltas sin acoplo (solo si N>=2) se anaden como
          referencia fina y gris, para comparar contra el caso
          desacoplado."""
        fig = plt.Figure(figsize=(7.5, 5.5))
        ax = fig.subplots(1, 1)
        mode_ymax = 0.0

        if tube.N >= 2 and p["show_walls_ref"]:
            walls_labelled = set()
            for m in p["d_m_list"]:
                for j in range(tube.N):
                    w2 = np.array([tube.omega_j2(j, m, k) for k in k_au])
                    y = self._to_eV(w2)
                    mode_ymax = max(mode_ymax, np.nanmax(y))
                    lbl = None
                    if j not in walls_labelled:
                        lbl = rf"pared sola $a={tube.a[j] * BOHR_TO_NM:.3g}$ nm (sin acoplo)"
                        walls_labelled.add(j)
                    ax.plot(k_inv_nm, y, color="0.65", lw=1.0, ls=":",
                            alpha=0.9, label=lbl, zorder=1)

        for mi, m in enumerate(p["d_m_list"]):
            ls = LINESTYLES[mi % len(LINESTYLES)]
            if tube.N == 1:
                w2 = np.array([tube.omega_j2(0, m, k) for k in k_au])
                y = self._to_eV(w2)
                mode_ymax = max(mode_ymax, np.nanmax(y))
                ax.plot(k_inv_nm, y, color="black", ls=ls, lw=1.6,
                        label=rf"$m={m}$", zorder=2)
            else:
                branches = np.array([tube.branches(m, k) for k in k_au])
                for i in range(tube.N):
                    y = self._to_eV(branches[:, i])
                    mode_ymax = max(mode_ymax, np.nanmax(y))
                    ax.plot(k_inv_nm, y, color="black", ls=ls, lw=1.6,
                            label=rf"$m={m}$" if i == 0 else None, zorder=2)

        KV_COLORS = ["tab:blue", "tab:red", "tab:green", "tab:purple",
                     "tab:orange", "tab:brown"]
        for vi, v_over_c in enumerate(p["d_v_list"]):
            v_au = v_over_c * C_AU
            y = self._to_eV((k_au * v_au) ** 2)
            ax.plot(k_inv_nm, y, color=KV_COLORS[vi % len(KV_COLORS)], lw=1.4,
                    label=rf"$kv$ ($v={v_over_c:g}c$)", zorder=3)

        ax.set_xlabel(r"$k$ ($\mathrm{nm}^{-1}$)")
        ax.set_ylabel(r"Energia del plasmon: $\hbar\omega$ (eV)")
        ax.set_xlim(k_inv_nm.min(), k_inv_nm.max())
        if mode_ymax > 0 and np.isfinite(mode_ymax):
            ax.set_ylim(0, mode_ymax * 1.15)
        ax.legend(loc="best", fontsize=8, ncol=1)
        ax.grid(alpha=0.3)
        ax.set_title(f"Dispersion plasmonica -- {label}")
        fig.tight_layout()
        self._new_canvas(fig, "dispersion.png")

    def _on_save(self):
        if not self.figures:
            return
        directory = filedialog.askdirectory(title="Elige carpeta para guardar las figuras")
        if not directory:
            return
        saved = []
        for fig, name in self.figures:
            path = f"{directory}/{name}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            saved.append(path)
        self.status_var.set(f"Guardadas {len(saved)} figura(s) en {directory}")


if __name__ == "__main__":
    app = App()
    app.mainloop()
